# PANDA — Train all 5 folds + compute 5-fold mean QWK

Trains all 5 folds sequentially in one Kaggle session, then runs `src.oof` to compute the 5-fold mean QWK and write `oof_predictions.csv`.

**Inputs:** `panda-resized-train-data-512x512` (xhlulu)
**Settings:** GPU T4 ON, Internet ON
**Runtime:** ~30 min × 5 folds = ~2.5 hours

After commit, publish `/kaggle/working/efficientnetb0_fold{0..4}.pth` and `oof_predictions.csv` as a single Kaggle Dataset called `panda-effnetb0-5fold-baseline`.

In [ ]:
REPO   = 'https://github.com/Shashaboii/AIMI_Panda_Challenge.git'
BRANCH = 'main'
FOLDS  = [0, 1, 2, 3, 4]   # which folds to train; leave [1,2,3,4] if you already have fold 0
EPOCHS = 6

In [ ]:
import subprocess, os
if os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, '/kaggle/working/repo'], check=True)
subprocess.run(['git', '-C', '/kaggle/working/repo', 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
!pip install -q efficientnet_pytorch

In [ ]:
import glob

# Auto-detect the xhlulu image dir
candidates = (glob.glob('/kaggle/input/*resized*512*/train_images/train_images') +
              glob.glob('/kaggle/input/*resized*512*/train_images') +
              glob.glob('/kaggle/input/**/train_images', recursive=True))
IMAGE_DIR = next((c for c in candidates if os.path.isdir(c) and len(os.listdir(c)) > 5000), None)
if IMAGE_DIR is None:
    raise RuntimeError(f'Could not find image dir. Input contents: {os.listdir("/kaggle/input")}')
print('IMAGE_DIR:', IMAGE_DIR, 'files:', len(os.listdir(IMAGE_DIR)))

In [ ]:
# Train each fold. If a fold's weights already exist, skip it (you may have done fold 0 earlier).
import time

for fold in FOLDS:
    weight_path = f'/kaggle/working/efficientnetb0_fold{fold}.pth'
    if os.path.exists(weight_path):
        print(f'fold {fold}: weights already exist, skipping')
        continue
    print(f'\n=== Training fold {fold} ===')
    t0 = time.time()
    result = subprocess.run(
        ['python', '-m', 'src.train',
         '--fold', str(fold),
         '--folds-csv', '/kaggle/working/repo/data/train_folds.csv',
         '--image-dir', IMAGE_DIR,
         '--epochs', str(EPOCHS),
         '--output-dir', '/kaggle/working'],
        cwd='/kaggle/working/repo'
    )
    print(f'fold {fold} finished in {(time.time()-t0)/60:.1f} min, exit code {result.returncode}')
    if result.returncode != 0:
        raise RuntimeError(f'Training fold {fold} failed')

In [ ]:
# Compute OOF predictions and 5-fold mean QWK across all trained folds
result = subprocess.run(
    ['python', '-m', 'src.oof',
     '--folds-csv', '/kaggle/working/repo/data/train_folds.csv',
     '--image-dir', IMAGE_DIR,
     '--weights-dir', '/kaggle/working',
     '--output-csv', '/kaggle/working/oof_predictions.csv'],
    cwd='/kaggle/working/repo'
)
if result.returncode != 0:
    raise RuntimeError('OOF evaluation failed')

In [ ]:
# Inspect the OOF predictions — confusion matrix on the rounded predictions
import sys
sys.path.insert(0, '/kaggle/working/repo')
import pandas as pd
from src.eval import qwk, confusion_matrix_str

oof = pd.read_csv('/kaggle/working/oof_predictions.csv')
print(f'OOF rows: {len(oof)}')
print(f'Global OOF QWK: {qwk(oof.pred_raw.values, oof.isup_grade.values):.4f}')
print()
print('Confusion matrix (rows = true, cols = predicted):')
print(confusion_matrix_str(oof.pred_raw.values, oof.isup_grade.values))

In [ ]:
# Final list of outputs to publish as a Kaggle Dataset
for f in sorted(os.listdir('/kaggle/working')):
    p = f'/kaggle/working/{f}'
    if os.path.isfile(p):
        print(f'{f}  ({os.path.getsize(p)/1e6:.2f} MB)')